In [18]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MinMaxScaler


In [19]:
# Load the dataset
anime_df=pd.read_csv('anime.csv')


In [20]:
anime_df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [21]:
anime_df.describe

<bound method NDFrame.describe of        anime_id                                               name  \
0         32281                                     Kimi no Na wa.   
1          5114                   Fullmetal Alchemist: Brotherhood   
2         28977                                           Gintama°   
3          9253                                        Steins;Gate   
4          9969                                      Gintama&#039;   
...         ...                                                ...   
12289      9316       Toushindai My Lover: Minami tai Mecha-Minami   
12290      5543                                        Under World   
12291      5621                     Violence Gekiga David no Hoshi   
12292      6133  Violence Gekiga Shin David no Hoshi: Inma Dens...   
12293     26081                   Yasuji no Pornorama: Yacchimae!!   

                                                   genre   type episodes  \
0                   Drama, Romance, School, Super

In [22]:
anime_df.shape

(12294, 7)

In [23]:
# Check for missing values
print("Missing values before cleaning:\n", anime_df.isnull().sum())

# Drop rows where essential fields are missing
anime_df.dropna(subset=['name', 'genre', 'rating'], inplace=True)

# Fill missing 'episodes' with 0 and convert to numeric
anime_df['episodes'] = pd.to_numeric(anime_df['episodes'], errors='coerce').fillna(0)

# Reset index
anime_df.reset_index(drop=True, inplace=True)

print("\nAfter cleaning:")
print(anime_df.info())


Missing values before cleaning:
 anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

After cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12017 entries, 0 to 12016
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12017 non-null  int64  
 1   name      12017 non-null  object 
 2   genre     12017 non-null  object 
 3   type      12017 non-null  object 
 4   episodes  12017 non-null  float64
 5   rating    12017 non-null  float64
 6   members   12017 non-null  int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 657.3+ KB
None


In [24]:
# Combine text and numerical features into one column for similarity
anime_df['combined_features'] = (
    anime_df['genre'].fillna('') + ' ' +
    anime_df['type'].fillna('') + ' ' +
    anime_df['rating'].astype(str) + ' ' +
    anime_df['members'].astype(str)
)

anime_df.head(3)


,anime_id,name,genre,type,episodes,rating,members,combined_features
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1.0,9.37,200630,"Drama, Romance, School, Supernatural Movie 9.3..."
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665,"Action, Adventure, Drama, Fantasy, Magic, Mili..."
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.25,114262,"Action, Comedy, Historical, Parody, Samurai, S..."


In [25]:
# Convert genre text into a matrix of token counts
vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(anime_df['combined_features'])

# Compute cosine similarity
cosine_sim = cosine_similarity(count_matrix, count_matrix)


In [27]:
def recommend_anime(title, n=5):
    # Ensure case-insensitive match
    idx = anime_df[anime_df['name'].str.lower() == title.lower()].index
    
    if len(idx) == 0:
        return f"❌ Anime titled '{title}' not found in dataset."
    
    idx = idx[0]
    similarity_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    # Get top n similar animes (excluding itself)
    similar_anime_indices = [i[0] for i in similarity_scores[1:n+1]]
    
    return anime_df[['name', 'genre', 'rating']].iloc[similar_anime_indices]


In [28]:
# Example: recommend anime similar to "Naruto"
recommend_anime("Naruto", n=10)


,name,genre,rating
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94
1930,Dragon Ball Super,"Action, Adventure, Comedy, Fantasy, Martial Ar...",7.40
3037,Tenjou Tenge,"Action, Comedy, Ecchi, Martial Arts, School, S...",7.10
1573,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.50
206,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.32
515,Dragon Ball Kai (2014),"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.01
588,Dragon Ball Kai,"Action, Adventure, Comedy, Fantasy, Martial Ar...",7.95
1209,Medaka Box Abnormal,"Action, Comedy, Ecchi, Martial Arts, School, S...",7.63
2615,Medaka Box,"Action, Comedy, Ecchi, Martial Arts, School, S...",7.21
486,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",8.03


In [29]:
# Example: check multiple recommendations
titles_to_test = ["Death Note", "One Piece", "Attack on Titan"]
for title in titles_to_test:
    print(f"\n🎥 Recommendations for '{title}':")
    print(recommend_anime(title, n=5))



🎥 Recommendations for 'Death Note':
                              name  \
981                Mousou Dairinin   
144  Higurashi no Naku Koro ni Kai   
334      Higurashi no Naku Koro ni   
778             Death Note Rewrite   
445               Mirai Nikki (TV)   

                                                 genre  rating  
981  Drama, Mystery, Police, Psychological, Superna...    7.74  
144     Mystery, Psychological, Supernatural, Thriller    8.41  
334  Horror, Mystery, Psychological, Supernatural, ...    8.17  
778  Mystery, Police, Psychological, Supernatural, ...    7.84  
445  Action, Mystery, Psychological, Shounen, Super...    8.07  

🎥 Recommendations for 'One Piece':
                                                   name  \
231   One Piece: Episode of Merry - Mou Hitori no Na...   
241   One Piece: Episode of Nami - Koukaishi no Nami...   
896   One Piece: Episode of Sabo - 3 Kyoudai no Kizu...   
1930                                  Dragon Ball Super   
86           

####🧩 Interview Questions (for report)

Q1: What is the difference between user-based and item-based collaborative filtering?
A1:

User-based filtering recommends items liked by similar users.

Item-based filtering recommends items similar to what a user has liked.
Item-based is more stable because item relationships change less often than user preferences.

Q2: What is collaborative filtering and how does it work?
A2:
Collaborative filtering uses past user-item interactions to recommend new items. It assumes that users with similar preferences will like similar things. It works by building a similarity matrix between users or items, then using those similarities to predict ratings or recommend items.